In [ ]:
# W8 Day 6 - 完整链路图
# matplotlib 中文字体配置
from matplotlib import font_manager
import matplotlib.pyplot as plt
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print(f"中文字体配置完成: {font_name}")

# 🧱 LangChat 心智模型｜第8周-Day6：走完整条链

> **⚡ 动手交付：画一张完整链路图，标出每一步的输入输出**
>
> **日期**：2026-07-25（周六）
>
> **今日核心问题：为什么这条链不能短一步？**

## 📅 学习进度

```
W1  ████████████████████ ✅ Transformer与大模型训练
W2  ████████████████████ ✅ 微调与RLHF
W3  ████████████████████ ✅ RAG与知识增强
W4  ████████████████████ ✅ 推理与思维链
W5  ████████████████████ ✅ Agent与工具使用
W6  ████████████████████ ✅ LLM Agent实战
W7  ████████████████████ ✅ 数字员工架构深化
W8  █████████████████░░░ 🔥 LangChat链路全景 (Day6/7)
W9  ░░░░░░░░░░░░░░░░░░░░ 📝 Domain Deep Dive
W10 ░░░░░░░░░░░░░░░░░░░░ 🔒 Governance
W11 ░░░░░░░░░░░░░░░░░░░░ 📊 Code Reality
```

**进度: 8/13 周 (61.5%) | Day 40/91**

# 🔄 往期回顾（W8 链路前五天）

## W8 本周已学

| Day | 链路站点 | 核心问题 | 与今天的关系 |
|-----|---------|----------|-------------|
| Day1 | Agent Host → LangChat | 为什么 LangChat 不是 Agent Host？ | 链路起点 |
| Day2 | ApplicationContract | 为什么 Contract 不是 API 文档？ | 业务契约层 |
| Day3 | Blueprint → Compiler → ExecutionPlan | 为什么 Blueprint 不能直接运行？ | 编译链路 |
| Day4 | Runtime 无状态执行 | 为什么 Runtime 不保存状态？ | 执行环境 |
| Day5 | Capability + Connector | 为什么 Capability 不是 Plugin？ | 能力与连接 |

## 💡 今天的目标

把五天的碎片拼成一张完整的图：**10个站点 · 7个治理检查点 · 0步可缩短**

# 📚 Part 1：全景链路图

## LangChat 完整执行链路（从 Agent Host 到业务结果）

```
  ① Agent Host          ② Gateway           ③ SkillRelease
     用户意图        →    六维身份解析    →    发现 + 资格检查
         │                   │                    │
  ④ 幂等 & 限流         ⑤ HITL 审批          ⑥ 创建执行记录
     防重复 + 防刷      →   人审门控        →   (execution记录)
         │                   │                    │
  ⑦ Read-Only 守卫      ⑧ 分发执行           ⑨ SkillRelease Executor
     写操作检测       →   调度执行器       →   (如 W09 知识库问答)
         │                   │                    │
  ⑩ 七字段响应
     结构化返回 → Agent Host
```

## 治理检查点

| CP | 名称 | 文件 | 检查内容 |
|----|------|------|----------|
| CP-1 | 六维身份 | `six_dim_context.py` | client/actor/tenant/workspace/scope/delegation |
| CP-2 | SkillRelease 资格 | `eligibility.py` | scope 覆盖 + lifecycle=published |
| CP-3 | 幂等重放 | `execution_replay.py` | Idempotency-Key 匹配 |
| CP-4 | 速率限制 | `canonical_rate_limit.py` | RPM ≤ 300 |
| CP-5 | HITL 审批 | `execution_service.py` | review_assignee 存在 |
| CP-6 | 只读守卫 | `read_only_guard.py` | effect_policy + 递归扫描写指标 |
| CP-7 | 执行器存在 | `execution_dispatch.py` | executor_fn 已注册 |

# 📚 Part 2：逐站解析

## 站点 ① Agent Host → LangChat

- **输入**：用户消息（"报销流程是什么？"）
- **输出**：`POST /v1/skill-releases/{skill_id}/invoke` + 六维身份头
- **代码**：`canonical/router.py` → `handle_canonical_invoke()`

## 站点 ② 六维身份解析

- **输入**：HTTP 请求头
- **输出**：`SixDimExecutionContext` (frozen dataclass)
- **代码**：`server/auth/six_dim_context.py` → `get_six_dim_context()`
- **关键**：ADR-001 §8 "不可只信任请求体声明"

## 站点 ③ SkillRelease 资格检查

- **输入**：skill_id + SixDimExecutionContext
- **输出**：CanonicalPreparedExecution（Descriptor + input_hash）
- **代码**：`execution_preparation.py` → `prepare_canonical_execution()`

## 站点 ④ 幂等 + 限流

- **输入**：Idempotency-Key + credential_id
- **输出**：放行或重放历史结果
- **代码**：`execution_replay.py` + `canonical_rate_limit.py`

## 站点 ⑤ HITL 门控

- **输入**：human_review_gate + review_assignee 配置
- **输出**：放行 / 返回 202 待审 + review_token
- **代码**：`execution_service.py` → `get_review_assignee()`

## 站点 ⑥ 执行记录

- **输入**：六维上下文 + Descriptor + 输入数据
- **输出**：execution_id（持久化审计记录）
- **代码**：`canonical_execution_repository.py` → `create_execution()`

## 站点 ⑦ Read-Only 守卫

- **输入**：SkillReleaseDescriptor
- **输出**：安全放行 / ReadOnlyViolationError
- **代码**：`read_only_guard.py` → `enforce_read_only()`
- **机制**：递归扫描 workflow_binding（8层深度）查找 _WRITE_INDICATORS

## 站点 ⑧ 分发执行

- **输入**：CanonicalDispatchCommand
- **输出**：七字段结果 / CanonicalDispatchFailed
- **代码**：`execution_dispatch.py` → `dispatch_canonical_execution()`
- **追踪**：OpenTelemetry span `skill_release.invoke:{skill_id}`

## 站点 ⑨-⑩ 执行 + 七字段响应

- **七字段**：summary / details / references / assumptions / human_review_required / next_actions / confidence
- **代码**：`descriptor.py` → `_SEVEN_FIELD_OUTPUT_SCHEMA`

# 📚 Part 3：为什么这条链不能短一步？

| 去掉 | 后果 |
|------|------|
| ① Agent Host 直连 | LangChat 变成 AI 聊天产品，不再是能力平台 |
| ② 六维身份 | 无法做租户隔离、scope 校验、委托链追踪 |
| ③ 资格检查 | 任何调用者能访问任何技能（含其他租户专属技能） |
| ④ 幂等+限流 | 重复执行 + LLM 推理配额耗尽 |
| ⑤ HITL 审批 | AI 直接披露敏感信息（薪资、人事） |
| ⑥ 执行记录 | 无审计追踪，出了问题无法回查 |
| ⑦ 只读守卫 | 写操作（db_write/http_request）在 P0 阶段被执行 |
| ⑧ 分发执行 | Executor 未注册时系统 500 崩溃 |

**结论：每一步有独立的治理目的，链路不可缩短。**

In [ ]:
# 可运行可视化 1：LangChat 完整执行链路流程图
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

stations = [
    ("① Agent Host\n用户意图", "#4CAF50"),
    ("② 六维身份\n身份解析", "#2196F3"),
    ("③ 资格检查\nScope+生命周期", "#2196F3"),
    ("④ 幂等+限流\n防重复+防刷", "#FF9800"),
    ("⑤ HITL 门控\n人审检查", "#FF9800"),
    ("⑥ 执行记录\n审计留痕", "#9C27B0"),
    ("⑦ 只读守卫\n写操作检测", "#F44336"),
    ("⑧ 分发执行\n调度Executor", "#4CAF50"),
    ("⑨ 技能执行\nWorkflow+知识库", "#4CAF50"),
    ("⑩ 七字段响应\n结构化返回", "#4CAF50"),
]

fig, ax = plt.subplots(figsize=(20, 6))
ax.set_xlim(-1, 21)
ax.set_ylim(-2, 4)
ax.axis('off')
ax.set_title('LangChat 完整执行链路：从用户意图到业务结果\n（每一步不可缩短）',
             fontsize=16, fontweight='bold', pad=20)

for i, (label, color) in enumerate(stations):
    x = i * 2.2
    rect = mpatches.FancyBboxPatch((x-0.8, 0.5), 1.6, 2.0,
                                     boxstyle="round,pad=0.1",
                                     facecolor=color, edgecolor='black',
                                     alpha=0.85, linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, 1.5, label, ha='center', va='center',
            fontsize=8, fontweight='bold', color='white')
    if i < len(stations) - 1:
        ax.annotate('', xy=(x+2.2-0.8, 1.5), xytext=(x+0.8, 1.5),
                    arrowprops=dict(arrowstyle='->', color='#333', lw=2))

checkpoints = [
    (1, "CP-1\n身份", "#2196F3"),
    (2, "CP-2\n资格", "#2196F3"),
    (3, "CP-3/4\n幂等+限流", "#FF9800"),
    (4, "CP-5\n人审", "#FF9800"),
    (5, "CP-6\n审计", "#9C27B0"),
    (6, "CP-7\n只读", "#F44336"),
]

for idx, label, color in checkpoints:
    x = idx * 2.2
    ax.text(x, -0.5, label, ha='center', va='top',
            fontsize=7, color=color, fontweight='bold')
    ax.annotate('', xy=(x, -0.1), xytext=(x, 0.4),
                arrowprops=dict(arrowstyle='->', color=color, lw=1, ls='--'))

legend_elements = [
    mpatches.Patch(facecolor='#4CAF50', label='执行层（业务逻辑）'),
    mpatches.Patch(facecolor='#2196F3', label='认证与资格（治理）'),
    mpatches.Patch(facecolor='#FF9800', label='限流与人审（治理）'),
    mpatches.Patch(facecolor='#9C27B0', label='审计追踪（治理）'),
    mpatches.Patch(facecolor='#F44336', label='安全底线（治理）'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=9,
          ncol=5, bbox_to_anchor=(0, 1.15))

ax.text(10, -1.5, '10个站点 · 7个治理检查点 · 0步可缩短',
        ha='center', fontsize=12, fontweight='bold', color='#333',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#FFF9C4', edgecolor='#FBC02D'))

plt.tight_layout()
plt.savefig('w8d6_chain_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ 链路流程图已生成")

In [ ]:
# 可运行可视化 2：治理检查点 × 维度覆盖热力图
import matplotlib.pyplot as plt
import numpy as np

dimensions = ['身份认证', '访问控制', '数据安全', '审计追踪', '可靠性', '性能保护']
checkpoints = ['CP-1\n六维身份', 'CP-2\n资格检查', 'CP-3\n幂等', 'CP-4\n限流',
               'CP-5\nHITL', 'CP-6\n只读守卫', 'CP-7\n执行器']

coverage = np.array([
    [2, 1, 1, 2, 0, 0],
    [0, 2, 1, 1, 0, 0],
    [0, 0, 0, 1, 2, 0],
    [0, 0, 0, 0, 1, 2],
    [0, 2, 2, 1, 0, 0],
    [0, 0, 2, 1, 1, 0],
    [0, 0, 0, 1, 2, 0],
])

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(coverage, cmap='YlOrRd', aspect='auto', vmin=0, vmax=2)

ax.set_xticks(range(len(dimensions)))
ax.set_yticks(range(len(checkpoints)))
ax.set_xticklabels(dimensions, fontsize=10)
ax.set_yticklabels(checkpoints, fontsize=10)

labels = {0: '—', 1: '◐', 2: '●'}
for i in range(len(checkpoints)):
    for j in range(len(dimensions)):
        ax.text(j, i, labels[coverage[i, j]], ha='center', va='center',
                fontsize=14, fontweight='bold',
                color='white' if coverage[i,j] == 2 else '#333')

ax.set_title('LangChat 治理检查点 × 维度覆盖热力图\n（●=完全覆盖  ◐=部分覆盖  —=不覆盖）',
             fontsize=13, fontweight='bold', pad=15)
plt.colorbar(im, ax=ax, shrink=0.8, label='覆盖程度')

total_checks = coverage.size
full_coverage = (coverage == 2).sum()
partial = (coverage == 1).sum()
ax.text(0.5, -0.12, f'完全覆盖: {full_coverage}/{total_checks}  |  部分覆盖: {partial}/{total_checks}  |  最大覆盖维度: 审计追踪',
        transform=ax.transAxes, ha='center', fontsize=10, style='italic')

plt.tight_layout()
plt.savefig('w8d6_governance_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ 治理热力图已生成")

In [ ]:
# 可运行可视化 3：链路数据流图（每一步的输入与输出）
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(18, 12))
ax.set_xlim(0, 18)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('LangChat 链路数据流：每一步的输入与输出\n（从 HTTP 请求到七字段响应）',
             fontsize=14, fontweight='bold', pad=15)

flow_data = [
    (9, 11, "HTTP 请求\nPOST /v1/skill-releases/{id}/invoke\n+ 六维身份头", "#E8F5E9", "#4CAF50"),
    (9, 9.2, "SixDimExecutionContext (frozen)\nclient/actor/tenant/workspace/scope/delegation", "#E3F2FD", "#2196F3"),
    (9, 7.4, "CanonicalPreparedExecution\nDescriptor + input_data + input_hash", "#E3F2FD", "#2196F3"),
    (9, 5.6, "Execution Record\nexecution_id + 六维上下文 + effect_policy", "#F3E5F5", "#9C27B0"),
    (9, 3.8, "CanonicalDispatchCommand\nDescriptor + 输入数据 + runtime_context", "#FFF3E0", "#FF9800"),
    (9, 2.0, "七字段结构化输出\nsummary/details/references/assumptions/\nhuman_review_required/next_actions/confidence", "#E8F5E9", "#4CAF50"),
]

governance_labels = [
    (2, 9.2, "🔒 CP-1\n六维身份\n不可只信任\n请求体声明", "#2196F3"),
    (2, 7.4, "🔒 CP-2/3/4\n资格 + 幂等\n+ 限流", "#FF9800"),
    (2, 5.6, "🔒 CP-5/6\nHITL +\n只读守卫", "#F44336"),
    (2, 3.8, "🔒 CP-7\n执行器\n存在性校验", "#9C27B0"),
]

for i, (x, y, text, bg, border) in enumerate(flow_data):
    rect = mpatches.FancyBboxPatch((x-3.5, y-0.6), 7, 1.2,
                                     boxstyle="round,pad=0.15",
                                     facecolor=bg, edgecolor=border, linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=7.5,
            fontweight='bold', color='#333')
    if i < len(flow_data) - 1:
        ax.annotate('', xy=(x, flow_data[i+1][1]+0.6), xytext=(x, y-0.6),
                    arrowprops=dict(arrowstyle='->', color='#666', lw=2.5))

for x, y, text, color in governance_labels:
    rect = mpatches.FancyBboxPatch((x-1.2, y-0.55), 2.4, 1.1,
                                     boxstyle="round,pad=0.1",
                                     facecolor='white', edgecolor=color,
                                     linewidth=1.5, linestyle='--')
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=7,
            fontweight='bold', color=color)
    ax.annotate('', xy=(5.5, y), xytext=(x+1.2, y),
                arrowprops=dict(arrowstyle='->', color=color, lw=1, ls='--'))

right_labels = [
    (15.5, 11, "Agent Host\n发起", "#4CAF50"),
    (15.5, 9.2, "Gateway 层\n解析", "#2196F3"),
    (15.5, 7.4, "Preparation\n准备", "#2196F3"),
    (15.5, 5.6, "Audit\n记录", "#9C27B0"),
    (15.5, 3.8, "Dispatch\n调度", "#FF9800"),
    (15.5, 2.0, "Response\n返回", "#4CAF50"),
]
for x, y, text, color in right_labels:
    ax.text(x, y, text, ha='center', va='center', fontsize=8,
            fontweight='bold', color=color,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#FAFAFA',
                      edgecolor=color, alpha=0.8))

plt.tight_layout()
plt.savefig('w8d6_data_flow.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ 数据流图已生成")

In [ ]:
# 可运行可视化 4：链路完整性矩阵（Gap Analysis）
import matplotlib.pyplot as plt
import numpy as np

stations = ['① Agent Host', '② 六维身份', '③ 资格检查', '④ 幂等+限流',
            '⑤ HITL 门控', '⑥ 执行记录', '⑦ 只读守卫', '⑧ 分发执行',
            '⑨ Workflow执行', '⑩ Connector', 'v2 制品链', 'E3/E4 遗留']

status_scores = [3, 3, 3, 3, 3, 3, 3, 3, 2, 0, 0, 1]
colors = ['#4CAF50', '#4CAF50', '#4CAF50', '#4CAF50',
          '#4CAF50', '#4CAF50', '#4CAF50', '#4CAF50',
          '#FF9800', '#F44336', '#F44336', '#FF9800']
status_labels = ['🟢 对齐', '🟢 对齐', '🟢 对齐', '🟢 对齐',
                 '🟢 对齐', '🟢 对齐', '🟢 对齐', '🟢 对齐',
                 '🟡 过渡', '🔴 Gap', '🔴 Gap', '🟡 待迁移']

fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.barh(range(len(stations)), status_scores, color=colors, edgecolor='#333', linewidth=0.8)

ax.set_yticks(range(len(stations)))
ax.set_yticklabels(stations, fontsize=10)
ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(['❌ Gap', '🟡 待迁移', '🟡 过渡态', '🟢 已对齐'], fontsize=10)
ax.set_title('LangChat 链路完整性矩阵：目标态 vs 代码现实', fontsize=14, fontweight='bold', pad=15)
ax.set_xlim(0, 3.5)

for i, (bar, label) in enumerate(zip(bars, status_labels)):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            label, va='center', fontsize=9, fontweight='bold')

ax.axvline(x=2.5, color='#4CAF50', linestyle='--', alpha=0.5, linewidth=1)
ax.text(2.5, len(stations)-0.5, '  当前水平', fontsize=8, color='#4CAF50', style='italic')

plt.tight_layout()
plt.savefig('w8d6_gap_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gap 矩阵已生成")

# 📚 Part 4：Gap Analysis

| 链路站 | 状态 | Gap 说明 |
|--------|------|----------|
| ① - ⑧ 核心执行 | 🟢 已对齐 | canonical 执行链路完整可用 |
| ⑨ Workflow 执行 | 🟡 过渡态 | 目标态为 ExecutionPlanIR（v2 制品链） |
| ⑩ Connector | 🔴 最大 Gap | MCP 嵌在 Workflow 内，未独立治理 |
| v2 制品链 | 🔴 未实现 | Blueprint → Build → IR → SR v2 |
| E3/E4 遗留路径 | 🟡 待迁移 | SPA workflow / Public Chat |

## 💡 业务关联（LangChat × MallSenseAI）

当 MallSenseAI 作为行业能力包接入 LangChat 时，也需要走这条链路：
- MallSenseAI 的 Vision Capability 注册为 LangChat Capability（受治理描述符）
- 通过 SkillRelease 绑定为可执行技能（如"巡检分析"skill）
- Agent Host 调用该技能时，同样经过 10 站链路 + 7 个检查点
- Connector 连接 MallSenseAI 的检测器/GPU 推理服务

**链路的通用性保证了行业扩展的安全性。**

# 💡 今天多理解了什么

| # | 以前以为 | 现在知道 |
|---|---------|----------|
| 1 | 各模块是并列组件 | 是一条不可缩短的串行链路 |
| 2 | 治理检查点散落各处 | 画出来发现有 7 个 CP，覆盖 6 个维度 |
| 3 | read_only_guard 是小守卫 | 是 P0 最后的安全防线，递归 8 层扫描 |
| 4 | 幂等和限流是运维功能 | 是链路治理的一部分（CP-3/CP-4） |
| 5 | 当前代码接近完整 | 核心链路对齐，但 Connector + v2 制品链是 Gap |
| 6 | 链路图是架构师才画的 | 画图过程本身就是认知整理过程 |

# 🔮 重新设计时是否仍这样做？

## 会更早做的
1. **Connector 独立治理层** — 不嵌在 Workflow 内
2. **v2 制品链从 P0 开始建** — 不走 WorkflowSpec 弯路
3. **ApplicationContract 在 P0 引入** — 不让 SkillReleaseDescriptor 承担三个角色

## 不会改变的
1. **六维身份作为第一步** — 没有身份就没有治理
2. **Read-Only 守卫作为最后防线** — 纵深防御原则
3. **七字段结构化输出** — 给 Agent Host 提供可解析标准格式
4. **幂等 + 限流在准备阶段** — 在执行前做防重复和防刷

# 📝 Daily Engineering Log

## 2026-07-25（Week8-Day6）

### 新增
- 完整链路有 10 个站点、7 个治理检查点、覆盖 6 个治理维度
- 每一步有独立的治理目的，去掉任何一步都会打开具体缺口
- SixDimExecutionContext 是贯穿全链路的数据结构

### 修改
- 各模块是"组件图"（并列） → 是"链路图"（串行，不可跳过）
- read_only_guard 是"配置校验" → 是运行时递归扫描机制（8层深度）

### 确认
- 核心执行链路（① - ⑧）已完全对齐 ADR 目标态
- OpenTelemetry span 追踪已嵌入 dispatch 阶段

### 遗留
- Connector 独立治理是最大架构 Gap（P0 危险）
- v2 制品链完全未实现
- E3/E4 遗留路径未迁移
- D1 unified delegation wire profile 未冻结

### 技术债
- `_WRITE_INDICATORS` 枚举式检测不完备
- WorkflowSpec 需要在 cutover 阶段逐步替换
- Connector 没有 independent lifecycle

### 下一步
- 明天 Day7 Virtual CTO Review：五维评分
- Week 9 进入 Domain Deep Dive
- 关注 ADR-005 和 ADR-007 的工程推进

# 📖 术语表

| 英文 | 音标 | 中文 | 说明 |
|------|------|------|------|
| Canonical | /kəˈnɒnɪkəl/ | 规范的 | 标准执行路径 |
| Checkpoint | /ˈtʃekpɔɪnt/ | 检查点 | 链路上的治理校验节点 |
| Eligibility | /ˌelɪdʒəˈbɪləti/ | 资格 | Scope + 生命周期 + 可见性校验 |
| Idempotency | /ˌaɪdəmˈpɒtənsi/ | 幂等性 | 同一请求重复执行产生相同结果 |
| Dispatch | /dɪˈspætʃ/ | 分发 | 根据技能 ID 找到对应执行器并调用 |
| HITL | /eɪtʃ-tiː-el/ | 人工审核 | Human-In-The-Loop |
| Rate Limit | /reɪt ˈlɪmɪt/ | 速率限制 | 防止单一调用者耗尽资源 |
| Read-Only Guard | /riːd-ˈəʊnli ɡɑːd/ | 只读守卫 | P0 阶段阻断所有写操作 |
| SixDimExecutionContext | — | 六维执行上下文 | 携带六维身份信息 |
| Seven-Field Output | — | 七字段输出 | 标准结构化返回格式 |
| Chain Integrity | /tʃeɪn ɪnˈteɡrəti/ | 链路完整性 | 链路上无断裂、无旁路 |
| Depth Defense | /depθ dɪˈfens/ | 纵深防御 | 多层安全检查 |
| Effect Policy | /ɪˈfekt ˈpɒləsi/ | 效果策略 | read_only 或 conditional_write |
| Review Assignee | /rɪˈvjuː əˈsaɪniː/ | 审批指派人 | HITL 技能配置的审批人 |
| Delegation Chain | /ˌdelɪˈɡeɪʃən tʃeɪn/ | 委托链 | actor 代表另一主体的授权链 |
| Cutover | /ˈkʌtoʊvər/ | 切换 | 从旧系统迁移到新系统 |
| Gap Analysis | /ɡæp əˈnæləsɪs/ | 差距分析 | 目标态与代码现实的差距评估 |

# ❓ 课堂练习与课后测试

## 课堂练习

**练习 1**：凭记忆画出 LangChat 完整链路图（≥8 站点，≥5 检查点）。

**练习 2**：观察链路图，找出最大 Gap，如果你是 CTO 优先修哪个？

**练习 3**：选一个检查点，写 100 字分析去掉它的后果。

## 课后测试

**Q1**：链路中有多少个治理检查点？
- A. 3 个  B. 5 个  C. 7 个  D. 10 个

**Q2**：`enforce_read_only()` 的递归扫描深度是多少？
- A. 3 层  B. 5 层  C. 8 层  D. 无限制

**Q3**：哪个检查点覆盖了最多的治理维度？
- A. CP-1 六维身份  B. CP-3 幂等  C. CP-6 只读守卫  D. CP-7 执行器

**Q4**：当前链路上最大的 Gap 是什么？
- A. 六维身份不完整  B. Connector 未独立治理  C. 幂等有 bug  D. 限流太严

**Q5**：为什么说链路"不能短一步"？以下哪个不是原因？
- A. 每步有独立治理目的  B. 去掉打开安全缺口  C. 步骤多显得专业  D. 纵深防御要求多层检查

# 📚 真实参考

## ADR 文档

| 文档 | 关键章节 | 链路位置 |
|------|---------|---------|
| ADR-001 | §4 直连定位、§6 控制面/执行面、§8 六维身份 | ①②⑥⑦ |
| ADR-002 | D1 统一委托 wire profile | ② |
| ADR-003 | §13 SkillRelease API wire profile | ①③④⑤⑧ |
| ADR-004 | §4.1.1 遗留路径、§8 Connector 边界 | ⑩ |
| ADR-005 | D-1~D-5 制品链 + WorkflowSpec 退役 | ⑨ |
| ADR-007 | D-4 FrozenExecutionContext wire | ② |

## 代码文件

| 文件 | 路径 | 链路位置 |
|------|------|---------|
| Canonical Router | `skill_release/canonical/router.py` | ① HTTP 入口 |
| 六维上下文 | `server/auth/six_dim_context.py` | ② 身份解析 |
| 执行准备 | `skill_release/canonical/execution_preparation.py` | ③④ 准备 |
| 幂等重放 | `skill_release/canonical/execution_replay.py` | ④ 幂等 |
| 速率限制 | `skill_release/canonical/canonical_rate_limit.py` | ④ 限流 |
| 执行服务 | `skill_release/canonical/execution_service.py` | ⑤⑥ 编排 |
| 只读守卫 | `skill_release/canonical/read_only_guard.py` | ⑦ 安全底线 |
| 分发执行 | `skill_release/canonical/execution_dispatch.py` | ⑧ 调度 |
| W09 绑定 | `skill_release/bindings/w09.py` | ⑨ 执行器 |
| 描述符 | `skill_release/descriptor.py` | ⑨ 七字段 Schema |
| 执行契约 | `skill_release/canonical/execution_contracts.py` | 全链路数据结构 |